In [8]:
import pandas as pd
import re

In [19]:
df = pd.read_csv(r"/content/dataset_train.csv")

In [41]:
df = df[["id_str","full_text", "image_url", "tweet_url", "created_at"]]

In [43]:
#get random 10
sample = df.sample(n=10)
sample

,id_str,full_text,image_url,tweet_url,created_at
35640,1890477819823558883,what is vibe coding now?,NaN,https://x.com/duttasomrattwt/status/1890477819...,2025-02-14 19:06:50+00:00
31577,1991907440422768905,I Need Coffee - Episode 190 - Weekly BC Review...,NaN,https://x.com/steveendow/status/19919074404227...,2025-11-21 16:31:56+00:00
26363,1924241149033648152,People hate on Vibe Coding because it’s a tren...,NaN,https://x.com/MrsApers/status/1924241149033648152,2025-05-18 23:10:15+00:00
16875,1918418806948372862,Vibe coding more like ADHD coding\n\nI find my...,NaN,https://x.com/robj3d3/status/1918418806948372862,2025-05-02 21:34:20+00:00
13023,1910694405125841134,⛔ No Coding Required ⛔\n\nNetMind XYZ uses Nat...,https://pbs.twimg.com/media/GoQmbxEbYAQ-Ecq.jpg,https://x.com/NetMindXYZ/status/19106944051258...,2025-04-11 14:00:19+00:00
6053,1907794465416204437,GitHub Copilot and Tabnine are already making ...,NaN,https://x.com/rega_protocol/status/19077944654...,2025-04-03 13:57:00+00:00
24295,1905666197590360194,This guy made AI fix his production bugs for 2...,https://pbs.twimg.com/media/GnJJTmYWIAAIZ5r.jpg,https://x.com/Sabrina_Ramonov/status/190566619...,2025-03-28 17:00:01+00:00
37850,1946319105541435429,Everyone is embracing vibe coding as real soft...,NaN,https://x.com/Marcia_Ong/status/19463191055414...,2025-07-18 21:20:10+00:00
32675,1980675251655831964,A new vibe coding experience from Ai studio @G...,https://pbs.twimg.com/media/G3zFodpWYAAi_bv.jpg,https://x.com/Conor_D_Dart/status/198067525165...,2025-10-21 16:39:13+00:00
37818,1946346823536267414,this is vibecoding https://t.co/vsA4GgHHve,https://pbs.twimg.com/media/GwLQEL4XcAAuQzE.png,https://x.com/Thisistheaj/status/1946346823536...,2025-07-18 23:10:18+00:00


In [63]:
import html
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Pastikan sudah download resource nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('english')) # Atau 'indonesian' jika data campur

def preprocess_tweet_v2(text):
    # 1. Decode HTML Entities (Penting untuk X/Twitter data)
    # Ini akan merubah &lt; menjadi < sehingga re.sub simbol bisa bekerja
    text = html.unescape(text)

    text = text.lower()

    # 2. Pola Regex kamu sudah bagus, pertahankan untuk URL, Mentions, Hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    # 3. Handle spesifik Vibe Coding sebelum hapus simbol
    text = re.sub(r'vibe\s+coding', 'vibecoding', text)
    text = re.sub(r'vibe\s+code', 'vibecode', text)
    text = re.sub(r'vibe\s+coded', 'vibecoded', text)

    # 4. Hapus simbol dan angka
    text = re.sub(r'[^\w\s]', ' ', text) # Ganti simbol dengan spasi agar kata tidak menempel
    text = re.sub(r'\d+', '', text)

    # 5. Tokenisasi & Stopwords (The missing piece)
    tokens = word_tokenize(text)
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    #6. Pastikan cleaned_tokens tidak mengandung duplicate kata
    cleaned_tokens = list(dict.fromkeys(cleaned_tokens))

    return " ".join(cleaned_tokens)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [64]:
df_preprocessed = df['full_text'].fillna('').apply(preprocess_tweet_v2)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)
df_preprocessed = pd.concat([df_preprocessed, df['created_at']], axis=1)
df_preprocessed = pd.concat([df_preprocessed, df['tweet_url']], axis=1)
df_preprocessed = pd.concat([df_preprocessed, df['id_str']], axis=1)

In [75]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed = df_preprocessed.drop_duplicates(subset=['full_text'])
df_preprocessed['created_at'] = pd.to_datetime(df_preprocessed['created_at']).dt.strftime('%B %Y')
df_preprocessed.count()

/tmp/ipykernel_2034/642859057.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_preprocessed['created_at'] = pd.to_datetime(df_preprocessed['created_at']).dt.strftime('%B %Y')


,0
full_text,35159
image_url,16102
created_at,35159
tweet_url,35159
id_str,35159


In [83]:
#get sample of df preprocessed of 10 to print out
sample = df_preprocessed.sample(n=10)
sample

,full_text,image_url,created_at,tweet_url,id_str
34898,future web taught ever students often end lear...,https://pbs.twimg.com/media/G8nhkwRbEAAdrW2.jpg,December 2025,https://x.com/mdTarik1390649/status/2002379534...,2002379534117843448
1528,today pricing update far best affordable struc...,NaN,April 2025,https://x.com/robhou_/status/1914409884982108406,1914409884982108406
35850,vibecoding may trending experts agree real und...,NaN,September 2025,https://x.com/CSforALL/status/1964010821282853301,1964010821282853301
25226,someone work laughed mentioned vibecoding lite...,NaN,March 2025,https://x.com/GhwstVR/status/1903216146104320111,1903216146104320111
24633,trying vibecode react flow app either cursor dont,NaN,April 2025,https://x.com/abhiskrj/status/1910062124157878616,1910062124157878616
92,introducing jules coding agent powered gemini ...,https://pbs.twimg.com/amplify_video_thumb/1924...,May 2025,https://x.com/julesagent/status/19248902068531...,1924890206853116142
15319,allows users manage multiple agents simultaneo...,NaN,September 2025,https://x.com/marvefwesh/status/19722647739464...,1972264773946409387
27040,yesterday went tech week meet reminded sociali...,https://pbs.twimg.com/media/Gshn4ahXsAAgzac.jpg,June 2025,https://x.com/rudrank/status/1929906674535420347,1929906674535420347
13873,coding stress never sleeps developers creao co...,https://pbs.twimg.com/media/G9nin7oW8AAwh7D.jpg,January 2026,https://x.com/CryptoPope89448/status/200688430...,2006884301493686334
38420,finished vibecoding first app check,NaN,June 2025,https://x.com/aribmbang/status/193866485141467...,1938664851414679586


In [79]:
df_preprocessed.to_csv(r"./preprocessed_tweets.csv", index=False)